### Installation

[Traning](https://oneclickamd.ai/github/azheraly/Aqal-First-Urdu-Reasoning-Large-Language-Model/blob/main/nb/qwen2.5-CPT.ipynb)


In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
from huggingface_hub import notebook_login, list_repo_files, snapshot_download

notebook_login()

### Unsloth


In [2]:
%env UNSLOTH_RETURN_LOGITS = 1 # Run this to disable CCE since it is not supported for CPT

env: UNSLOTH_RETURN_LOGITS=1 # Run this to disable CCE since it is not supported for CPT


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Choose any! We auto support RoPE Scaling internally!
dtype = (
    None  # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
)
load_in_4bit = False  # Use 4bit quantization to reduce memory usage. Can be False.
load_in_8bit = False  # Use 8bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    # Can select any from the below:
    # "unsloth/Qwen2.5-0.5B", "unsloth/Qwen2.5-1.5B", "unsloth/Qwen2.5-3B"
    # "unsloth/Qwen2.5-14B",  "unsloth/Qwen2.5-32B",  "unsloth/Qwen2.5-72B",
    # And also all Instruct versions and Math. Coding versions!
    model_name="unsloth/Qwen3-8B",  # Choose ANY
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    load_in_8bit=load_in_8bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

We now add LoRA adapters so we only need to update 1 to 20% of all parameters!

We also add `embed_tokens` and `lm_head` to allow the model to learn out of distribution data.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=128,  # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
        "embed_tokens",
        "lm_head",
    ],  # Add for continual pretraining
    lora_alpha=32,
    lora_dropout=0,  # Supports any, but = 0 is optimized
    bias="none",  # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing="unsloth",  # True or "unsloth" for very long context
    random_state=3407,
    use_rslora=True,  # We support rank stabilized LoRA
    loftq_config=None,  # And LoftQ
)

<a name="Data"></a>

### Data Prep

We now use the Korean subset of the [Wikipedia dataset](https://huggingface.co/datasets/wikimedia/wikipedia) to first continually pretrain the model. You can use **any language** you like! Go to [Wikipedia's List of Languages](https://en.wikipedia.org/wiki/List_of_Wikipedias) to find your own language!

**[NOTE]** Remember to add the **EOS_TOKEN** to the tokenized output! Otherwise you'll get infinite generations!

**[NOTE]** Use https://translate.google.com to translate from English to Korean!


In [ ]:
from datasets import load_dataset

# English Prompt
# _wikipedia_prompt = """Wikipedia Article
# ### Title: {}

# ### Article:
# {}"""

ur_wiki_prompt = """ویکیپیڈیا آرٹیکل
### عنوان: {}

### مضمون:
{}"""


def load_dataset_by_name(
    dataset_name: str, language: str, tokenizer, split: str = "train"
):
    """
    Load a dataset by its name using the Hugging Face datasets library.
    Args:
      dataset_name (str): The name of the dataset to load.
      language: wikipedia dataset language
      tokenizer (Tokenizer): The tokenizer to use for processing the dataset.
      split (str): The split of the dataset to load (default is 'train').
    Returns:
        dataset: The loaded dataset.
    """

    try:
        dataset = load_dataset(dataset_name, language, split=split)

        # We select 1% of the data to make training faster!
        # dataset = dataset.train_test_split(train_size=0.01)["train"]

        EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN

        def formatting_prompts_func(examples):
            titles = examples["title"]
            texts = examples["text"]
            outputs = []
            for title, text in zip(titles, texts):
                # Must add EOS_TOKEN, otherwise your generation will go on forever!
                text = ur_wiki_prompt.format(title, text) + EOS_TOKEN
                outputs.append(text)
            return {
                "text": outputs,
            }

        dataset = dataset.map(
            formatting_prompts_func,
            batched=True,
        )

    except Exception as e:
        print(f"Error loading dataset {dataset_name}: {e}")
        raise RuntimeError(f"Failed to load dataset {dataset_name}")

    return dataset

In [ ]:
dataset = load_dataset_by_name("wikimedia/wikipedia", "20231101.ur", tokenizer)
print(f"Loaded dataset: {dataset}")

<a name="Train"></a>

### Continued Pretraining

Now let's use Unsloth's `UnslothTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 20 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

Also set `embedding_learning_rate` to be a learning rate at least 2x or 10x smaller than `learning_rate` to make continual pretraining work!


In [ ]:
from transformers import TrainingArguments
from unsloth import UnslothTrainer, UnslothTrainingArguments, is_bfloat16_supported

trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=4,
    args=UnslothTrainingArguments(
        per_device_train_batch_size=8,
        gradient_accumulation_steps=8,
        # Use warmup_ratio and num_train_epochs for longer runs!
        # max_steps=80,
        # warmup_steps=10,
        warmup_ratio=0.1,
        num_train_epochs=1,
        # Select a 2 to 10x smaller learning rate for the embedding matrices!
        learning_rate=5e-5,
        embedding_learning_rate=1e-5,
        logging_steps=100,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="Riazi-8B-CPT-Lora",  # Change this to your preferred output directory
        save_strategy="steps",
        save_steps=200,
        save_total_limit=2,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        report_to="none",  # Use wandb etc
        push_to_hub=True,
        hub_strategy="checkpoint",
    ),
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
# trainer_stats = trainer.train()

In [ ]:
print(f"Start training")

try:
    repo_id = "azherali/Riazi-8B-CPT-Lora"  # Change to your repo ID
    files = list_repo_files(repo_id)
    checkpoints = sorted(
        [f for f in files if f.startswith("last-checkpoint")],
    )
    if len(checkpoints) == 0:
        print("No checkpoints — starting new training.")
        trainer_stats = trainer.train()
    else:
        local_dir = snapshot_download(repo_id, allow_patterns="last-checkpoint/*")
        print("Resuming from:", local_dir)
        trainer_stats = trainer.train(
            resume_from_checkpoint=f"{local_dir}/last-checkpoint"
        )
except Exception as e:
    print("Repo not found or empty — starting fresh training.", e)
    trainer_stats = trainer.train()

<a name="Save"></a>

### Saving, loading finetuned models

To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!


In [ ]:
# Just LoRA adapters
if False:
    model.save_pretrained("Riazi-8B-CPT-Lora")  # Local saving
    tokenizer.save_pretrained("Riazi-8B-CPT-Lora")
if False:
    model.push_to_hub("azherali/Riazi-8B-CPT-Lora")
    tokenizer.push_to_hub("azherali/Riazi-8B-CPT-Lora")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.


In [ ]:
# Merge to 16bit
if False:
    model.save_pretrained_merged(
        "azherali/Riazi-8B-CPT",
        tokenizer,
        save_method="merged_16bit",
    )
if False:
    model.push_to_hub_merged(
        "azherali/Riazi-8B-CPT",
        tokenizer,
        save_method="merged_16bit",
    )

# Merge to 4bit
if False:
    model.save_pretrained_merged(
        "azherali/Riazi-8B-CPT",
        tokenizer,
        save_method="merged_4bit",
    )
if False:
    model.push_to_hub_merged(
        "azherali/Riazi-8B-CPT",
        tokenizer,
        save_method="merged_4bit",
    )

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:


In [ ]:
if False:
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="mistral_v0_lora",  # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length=max_seq_length,
        dtype=dtype,
        load_in_4bit=load_in_4bit,
    )
    FastLanguageModel.for_inference(model)  # Enable native 2x faster inference